# LightGCN Mathematics Without a Training Framework

This notebook shows what LightGCN computes before PyTorch and gradient descent are introduced.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from group_movie_recommender.algorithms.graph_data import (
    build_bipartite_graph,
    sample_bpr_batch,
)
from group_movie_recommender.algorithms.lightgcn_math import (
    bpr_loss,
    lightgcn_propagate,
    normalized_edge_weights,
    score_user_movie_pairs,
    split_user_movie_embeddings,
)

## 1. Build the same bipartite graph

Every edge is included in both directions. No rating value is used after the positive threshold has created the graph.

In [ ]:
train_edges = pd.DataFrame(
    {
        "userId": [20, 10, 10, 20, 30],
        "movieId": [200, 100, 200, 300, 100],
    }
)
graph = build_bipartite_graph(train_edges)
graph.edge_index

## 2. Normalize each neighbor contribution

For an edge from node `s` to node `t`, LightGCN uses

$$w_{s,t}=\frac{1}{\sqrt{deg(s)deg(t)}}.$$

Highly connected nodes therefore contribute less through each individual edge.

In [ ]:
edge_weights = normalized_edge_weights(
    graph.edge_index,
    num_nodes=graph.num_nodes,
)
pd.DataFrame(
    {
        "source": graph.edge_index[0],
        "target": graph.edge_index[1],
        "weight": edge_weights,
    }
)

## 3. Propagate and average embeddings

Layer zero is the trainable initial embedding. Each later layer contains normalized neighbor information. LightGCN averages all layers and uses no feature transformation or activation function.

In [ ]:
rng = np.random.default_rng(7)
initial_embeddings = rng.normal(
    loc=0.0,
    scale=0.1,
    size=(graph.num_nodes, 3),
)
propagation = lightgcn_propagate(
    initial_embeddings,
    graph.edge_index,
    num_layers=2,
)

print("Number of stored layers:", len(propagation.layer_embeddings))
display(pd.DataFrame(propagation.layer_embeddings[0]).add_prefix("initial_"))
display(pd.DataFrame(propagation.final_embeddings).add_prefix("final_"))

## 4. Score positive and negative movies

A user–movie score is the dot product of their final embeddings. BPR wants every sampled positive score to exceed the corresponding negative score.

In [ ]:
user_embeddings, movie_embeddings = split_user_movie_embeddings(
    propagation.final_embeddings,
    num_users=graph.num_users,
)
batch = sample_bpr_batch(graph, batch_size=8, random_seed=7)
positive_scores = score_user_movie_pairs(
    user_embeddings,
    movie_embeddings,
    batch["userIndex"].to_numpy(),
    batch["positiveMovieIndex"].to_numpy(),
)
negative_scores = score_user_movie_pairs(
    user_embeddings,
    movie_embeddings,
    batch["userIndex"].to_numpy(),
    batch["negativeMovieIndex"].to_numpy(),
)

scored_batch = batch.copy()
scored_batch["positiveScore"] = positive_scores
scored_batch["negativeScore"] = negative_scores
scored_batch["scoreDifference"] = positive_scores - negative_scores
scored_batch

In [ ]:
loss = bpr_loss(positive_scores, negative_scores)
loss

The embeddings are still random, so the loss is not expected to be good. Training will use automatic differentiation to update the initial embeddings and reduce this loss.

## Next step

Implement the same operations as a trainable PyTorch model, add mini-batch optimization, and verify that BPR loss decreases on a tiny graph before training on MovieLens 32M.